In [78]:
import pandas as pd
from collections import Counter
from graphviz import Digraph
import os

# 1. Thiết lập thư mục đầu ra
output_dir = "../results/phase_drift_analysis"
os.makedirs(output_dir, exist_ok=True)

# 2. Load và tiền xử lý dữ liệu
df = pd.read_csv("../data/bpi-challenge-2017/bpi_2017_cleaned.csv")
df = df[df["lifecycle:transition"] == "complete"]
df['time:timestamp'] = pd.to_datetime(df['time:timestamp'], format='ISO8601')
df['year'] = df['time:timestamp'].dt.year
df['week'] = df['time:timestamp'].dt.isocalendar().week
df = df.sort_values(['case:concept:name', 'time:timestamp'])

def build_dfg(df_sub):
    edges = Counter()
    for _, group in df_sub.groupby('case:concept:name'):
        activities = list(group['concept:name'])
        for i in range(len(activities) - 1):
            edges[(activities[i], activities[i+1])] += 1
    return edges

def draw_dfg(edges, title, file_path, threshold=5):
    dot = Digraph(comment=title)
    dot.attr(rankdir='LR', nodesep='0.4', ranksep='0.8', fontsize='12')
    
    # Lọc cạnh để sơ đồ nhỏ và rõ ràng
    edges = {k: v for k, v in edges.items() if v >= threshold}
    nodes = set([n for e in edges.keys() for n in e])
    
    for n in nodes:
        # Rút gọn tên để sơ đồ không bị quá rộng
        short_name = n.replace('Application', 'App').replace('Accepted', 'Acc')
        dot.node(n, short_name, shape='box', style='rounded,filled', fillcolor='#f9f9f9')
        
    for (src, tgt), freq in edges.items():
        dot.edge(src, tgt, label=str(freq), penwidth=str(0.5 + freq/200))
    
    dot.render(file_path, format='png', cleanup=True)

# 3. ĐỊNH NGHĨA 5 PHASE RỜI NHAU (DISJOINT PHASES)
# Cách chia này giúp so sánh "Trước và Sau" của từng sự kiện một cách tuyệt đối
phases = {
    "Phase_1_W1_18": df[df['week'] < 19],                            # Trước sự kiện 1
    "Phase_2_W19_23": df[(df['week'] >= 19) & (df['week'] < 24)],    # Sau sự kiện 1 (Tuần 19)
    "Phase_3_W24":    df[(df['week'] >= 24) & (df['week'] < 25)],    # Sự kiện bản lề (Tuần 24)
    "Phase_4_W25_48": df[(df['week'] >= 25) & (df['week'] < 49)],    # Sau sự kiện 2 (Tuần 25)
    "Phase_5_W49_End": df[df['week'] >= 49]                          # Cuối năm
}

# 4. Thực thi vẽ sơ đồ
for phase_name, subset in phases.items():
    print(f"Processing {phase_name}...")
    phase_path = os.path.join(output_dir, phase_name)
    os.makedirs(phase_path, exist_ok=True)
    
    for app_type in ['New credit', 'Limit raise']:
        app_label = app_type.replace(' ', '_').lower()
        app_subset = subset[subset['case:ApplicationType'] == app_type]
        
        if not app_subset.empty:
            edges = build_dfg(app_subset)
            # Threshold thấp vì dữ liệu mỗi phase đã bị chia nhỏ, giúp soi kỹ thay đổi
            draw_dfg(edges, f"{phase_name} - {app_type}", 
                     os.path.join(phase_path, f"{app_label}_dfg"),
                     threshold=15)

print(f"\n--- XONG! Sơ đồ đã được lưu tại: {output_dir} ---")

Processing Phase_1_W1_18...
Processing Phase_2_W19_23...
Processing Phase_3_W24...
Processing Phase_4_W25_48...
Processing Phase_5_W49_End...

--- XONG! Sơ đồ đã được lưu tại: ../results/phase_drift_analysis ---


In [1]:
import pandas as pd
import os
from graphviz import Digraph

# =========================================
# CONFIG
# =========================================
INPUT_FILE = "../data/bpi-challenge-2017/bpi_2017_cleaned.csv"
OUTPUT_DIR = "../results/trace_analysis"

TARGET_CASE_NUMS = [19576, 24076]

# =========================================
# 1. LOAD DATA
# =========================================
print("Loading data...")
df = pd.read_csv(INPUT_FILE)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], errors="coerce")
df = df.dropna(subset=["time:timestamp"])

# =========================================
# 2. PREPARE OUTPUT
# =========================================
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================
# 3. SORT DATA
# =========================================
df = df.sort_values(["case:concept:name", "time:timestamp"])

# =========================================
# 4. GET CASE IDS
# =========================================
case_ids = df["case:concept:name"].drop_duplicates().tolist()

print("Total cases:", len(case_ids))

# Convert index to actual case id
TARGET_CASES = []
for i in TARGET_CASE_NUMS:
    if i - 1 < len(case_ids):
        TARGET_CASES.append(case_ids[i - 1])
    else:
        print("Case index out of range:", i)

print("Target cases:", TARGET_CASES)

# =========================================
# 5. TRACE ANALYSIS (SINGLE CASE)
# =========================================
def analyze_trace(group, case_id):
    print("\n==============================")
    print("TRACE", case_id)
    print("==============================")

    group = group.sort_values("time:timestamp")

    # Flow
    flow = [
        f"{row['concept:name']}({row['lifecycle:transition']})"
        for _, row in group.iterrows()
    ]
    print("\nFlow:")
    print(" -> ".join(flow))

    # Loop detection
    activity_full = group["concept:name"] + "(" + group["lifecycle:transition"] + ")"
    counts = activity_full.value_counts()
    loops = counts[counts > 1]

    print("\nLoop:")
    if len(loops) > 0:
        print(loops)
    else:
        print("No loop")

    # Timestamp check
    if not group["time:timestamp"].is_monotonic_increasing:
        print("Timestamp ERROR")
    else:
        print("Timestamp OK")

    # Business logic
    activities = set(group["concept:name"])

    if "A_Accepted" not in activities:
        print("Missing A_Accepted")

    if "O_Sent" not in activities:
        print("Missing O_Sent")

    if "A_Cancelled" in activities:
        print("Case cancelled")

    if "A_Complete" in activities:
        print("Case completed")


# =========================================
# 6. DRAW TRACE GRAPH
# =========================================
def draw_trace_graph(group, case_id):
    dot = Digraph(comment=f"Trace {case_id}", format="png")

    group = group.sort_values("time:timestamp")
    events = group.reset_index(drop=True)

    for i, row in events.iterrows():
        activity = row["concept:name"]
        lifecycle = row["lifecycle:transition"]
        timestamp = row["time:timestamp"]

        label = f"{activity}\n({lifecycle})\n{timestamp}"

        color = "black"

        if lifecycle == "withdraw":
            color = "red"
        elif lifecycle == "schedule":
            color = "blue"
        elif lifecycle == "complete":
            color = "green"

        if activity == "A_Accepted":
            color = "darkgreen"
        elif activity == "A_Cancelled":
            color = "red"
        elif activity == "O_Sent":
            color = "orange"

        dot.node(str(i), label, color=color)

    for i in range(len(events) - 1):
        dot.edge(str(i), str(i + 1))

    safe_case_id = str(case_id).replace("/", "_")
    output_path = os.path.join(OUTPUT_DIR, safe_case_id)

    dot.render(output_path, view=False)
    print("Saved graph:", output_path)


# =========================================
# 7. BEFORE vs AFTER ANALYSIS
# =========================================
def analyze_before_after(df, case_ids, split_index, label):
    print("\n==============================")
    print("ANALYSIS AROUND", label)
    print("==============================")

    before_cases = case_ids[:split_index]
    after_cases = case_ids[split_index:]

    df_before = df[df["case:concept:name"].isin(before_cases)]
    df_after = df[df["case:concept:name"].isin(after_cases)]

    print("Before cases:", len(before_cases))
    print("After cases:", len(after_cases))

    # Average trace length
    len_before = df_before.groupby("case:concept:name").size().mean()
    len_after = df_after.groupby("case:concept:name").size().mean()

    print("\nAverage trace length:")
    print("Before:", round(len_before, 2))
    print("After :", round(len_after, 2))

    # Activity distribution
    act_before = df_before["concept:name"].value_counts(normalize=True)
    act_after = df_after["concept:name"].value_counts(normalize=True)

    diff = (act_after - act_before).fillna(0).sort_values(ascending=False)

    print("\nTop activity increase:")
    print(diff.head(10))

    print("\nTop activity decrease:")
    print(diff.tail(10))

    # Loop level
    def avg_loop(df_part):
        loops = []
        for _, g in df_part.groupby("case:concept:name"):
            max_loop = g["concept:name"].value_counts().max()
            loops.append(max_loop)
        return sum(loops) / len(loops)

    print("\nAverage loop level:")
    print("Before:", round(avg_loop(df_before), 2))
    print("After :", round(avg_loop(df_after), 2))

    # Duration
    def avg_duration(df_part):
        durations = []
        for _, g in df_part.groupby("case:concept:name"):
            d = g["time:timestamp"].max() - g["time:timestamp"].min()
            durations.append(d.total_seconds())
        return sum(durations) / len(durations)

    print("\nAverage duration (seconds):")
    print("Before:", round(avg_duration(df_before), 2))
    print("After :", round(avg_duration(df_after), 2))

    # Outcome rate
    def outcome_rate(df_part, activity):
        total = df_part["case:concept:name"].nunique()
        has_act = df_part[df_part["concept:name"] == activity]["case:concept:name"].nunique()
        return has_act / total if total > 0 else 0

    print("\nOutcome rate:")
    print("Accepted Before:", round(outcome_rate(df_before, "A_Accepted"), 3))
    print("Accepted After :", round(outcome_rate(df_after, "A_Accepted"), 3))

    print("Cancelled Before:", round(outcome_rate(df_before, "A_Cancelled"), 3))
    print("Cancelled After :", round(outcome_rate(df_after, "A_Cancelled"), 3))


# =========================================
# 8. RUN TRACE ANALYSIS
# =========================================
df_subset = df[df["case:concept:name"].isin(TARGET_CASES)]

for case_id, group in df_subset.groupby("case:concept:name"):
    analyze_trace(group, case_id)
    draw_trace_graph(group, case_id)

# =========================================
# 9. RUN DRIFT ANALYSIS
# =========================================
analyze_before_after(df, case_ids, 19576, "TRACE 19576")
analyze_before_after(df, case_ids, 24076, "TRACE 24076")

print("\nDone.")

Loading data...
Total cases: 31509
Target cases: ['Application_264466585', 'Application_54290162']

TRACE Application_264466585

Flow:
A_Create Application(complete) -> A_Submitted(complete) -> W_Handle leads(schedule) -> W_Handle leads(start) -> W_Handle leads(complete) -> W_Complete application(schedule) -> W_Complete application(start) -> A_Concept(complete) -> W_Complete application(suspend) -> A_Accepted(complete) -> O_Create Offer(complete) -> O_Created(complete) -> O_Create Offer(complete) -> O_Created(complete) -> O_Sent (mail and online)(complete) -> O_Sent (mail and online)(complete) -> W_Complete application(ate_abort) -> W_Call after offers(schedule) -> W_Call after offers(start) -> A_Complete(complete) -> W_Call after offers(suspend) -> W_Call after offers(resume) -> W_Call after offers(suspend) -> W_Call after offers(ate_abort) -> W_Validate application(schedule) -> W_Validate application(start) -> A_Validating(complete) -> W_Validate application(suspend) -> W_Validate ap